# Optimization with JAX

This notebook covers optimization techniques using JAX's automatic differentiation.

## Topics

1. Gradient descent and variants
2. Newton's method
3. Constrained optimization
4. Optimization with Optax
5. Root finding
6. Implicit differentiation

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit, hessian, jacfwd
from jax import lax
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)

## 1. Gradient Descent

The simplest optimization: follow the negative gradient.

$$x_{k+1} = x_k - \alpha \nabla f(x_k)$$

In [ ]:
def rosenbrock(x):
    """Rosenbrock function: classic optimization test.
    
    Minimum at (1, 1) with f(1,1) = 0.
    """
    return (1 - x[0])**2 + 100*(x[1] - x[0]**2)**2

def gradient_descent(f, x0, lr=0.001, n_steps=1000):
    """Basic gradient descent."""
    grad_f = jit(grad(f))
    x = x0
    history = [x]
    
    for _ in range(n_steps):
        x = x - lr * grad_f(x)
        history.append(x)
    
    return x, jnp.array(history)

x0 = jnp.array([-1.0, 1.0])
x_opt, history = gradient_descent(rosenbrock, x0, lr=0.001, n_steps=5000)

print(f"Initial: {x0}")
print(f"Final:   {x_opt}")
print(f"f(x_opt) = {rosenbrock(x_opt):.6f}")
print(f"Expected minimum: [1, 1]")

In [ ]:
# Visualize optimization path
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Contour plot
x_range = jnp.linspace(-2, 2, 100)
y_range = jnp.linspace(-1, 3, 100)
X, Y = jnp.meshgrid(x_range, y_range)
Z = jnp.array([[rosenbrock(jnp.array([x, y])) for x in x_range] for y in y_range])

axes[0].contour(X, Y, Z, levels=jnp.logspace(-1, 3, 20))
axes[0].plot(history[:, 0], history[:, 1], 'r.-', markersize=2, linewidth=0.5)
axes[0].plot(1, 1, 'g*', markersize=15, label='Minimum')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Optimization Path')
axes[0].legend()

# Loss curve
losses = [rosenbrock(x) for x in history]
axes[1].semilogy(losses)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('f(x)')
axes[1].set_title('Loss vs Iteration')

plt.tight_layout()
plt.show()

## 2. Momentum and Adam

Momentum accelerates convergence by accumulating gradients.

### Momentum
$$v_{k+1} = \beta v_k + \nabla f(x_k)$$
$$x_{k+1} = x_k - \alpha v_{k+1}$$

### Adam (Adaptive Moment Estimation)
Combines momentum with adaptive learning rates.

In [ ]:
def momentum_gd(f, x0, lr=0.001, beta=0.9, n_steps=1000):
    """Gradient descent with momentum."""
    grad_f = jit(grad(f))
    x = x0
    v = jnp.zeros_like(x)
    history = [x]
    
    for _ in range(n_steps):
        g = grad_f(x)
        v = beta * v + g
        x = x - lr * v
        history.append(x)
    
    return x, jnp.array(history)

def adam(f, x0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, n_steps=1000):
    """Adam optimizer."""
    grad_f = jit(grad(f))
    x = x0
    m = jnp.zeros_like(x)  # First moment
    v = jnp.zeros_like(x)  # Second moment
    history = [x]
    
    for t in range(1, n_steps + 1):
        g = grad_f(x)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        
        # Bias correction
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        
        x = x - lr * m_hat / (jnp.sqrt(v_hat) + eps)
        history.append(x)
    
    return x, jnp.array(history)

# Compare methods
x0 = jnp.array([-1.0, 1.0])

x_gd, hist_gd = gradient_descent(rosenbrock, x0, lr=0.001, n_steps=2000)
x_mom, hist_mom = momentum_gd(rosenbrock, x0, lr=0.001, n_steps=2000)
x_adam, hist_adam = adam(rosenbrock, x0, lr=0.01, n_steps=2000)

print("Final positions:")
print(f"  GD:       {x_gd}, f={rosenbrock(x_gd):.6f}")
print(f"  Momentum: {x_mom}, f={rosenbrock(x_mom):.6f}")
print(f"  Adam:     {x_adam}, f={rosenbrock(x_adam):.6f}")

## 3. Newton's Method

Use second-order information (Hessian) for faster convergence:

$$x_{k+1} = x_k - H^{-1} \nabla f(x_k)$$

Converges quadratically near the optimum!

In [ ]:
def newton_method(f, x0, n_steps=20):
    """Newton's method for optimization."""
    grad_f = jit(grad(f))
    hess_f = jit(hessian(f))
    x = x0
    history = [x]
    
    for _ in range(n_steps):
        g = grad_f(x)
        H = hess_f(x)
        
        # Newton step: solve H @ dx = -g
        dx = jnp.linalg.solve(H, -g)
        x = x + dx
        history.append(x)
    
    return x, jnp.array(history)

# Test on a simpler function (Newton struggles with Rosenbrock)
def quadratic(x):
    """f(x,y) = x² + 10y²"""
    return x[0]**2 + 10*x[1]**2

x0 = jnp.array([5.0, 5.0])
x_newton, hist_newton = newton_method(quadratic, x0, n_steps=10)

print("Newton's method on quadratic:")
for i, x in enumerate(hist_newton[:6]):
    print(f"  Step {i}: x = {x}, f(x) = {quadratic(x):.2e}")

print(f"\n✓ Converges in ~1 step for quadratic functions!")

## 4. Constrained Optimization

### 4.1 Penalty Method

Convert constraints to penalties:

$$\min f(x) \text{ s.t. } g(x) \leq 0 \Rightarrow \min f(x) + \mu \cdot \max(0, g(x))^2$$

In [ ]:
def constrained_example():
    """Minimize x² + y² subject to x + y >= 1."""
    
    def objective(x):
        return x[0]**2 + x[1]**2
    
    def constraint(x):
        # g(x) = 1 - x - y <= 0 (i.e., x + y >= 1)
        return 1 - x[0] - x[1]
    
    def penalized_objective(x, mu):
        penalty = jnp.maximum(0, constraint(x))**2
        return objective(x) + mu * penalty
    
    # Solve with increasing penalty
    x = jnp.array([0.0, 0.0])
    
    print("Penalty method optimization:")
    print(f"{'μ':>10} {'x':>20} {'f(x)':>10} {'g(x)':>10}")
    print("-" * 55)
    
    for mu in [1.0, 10.0, 100.0, 1000.0]:
        # Optimize with this penalty
        for _ in range(500):
            g = grad(lambda z: penalized_objective(z, mu))(x)
            x = x - 0.01 * g
        
        print(f"{mu:>10.0f} [{x[0]:.4f}, {x[1]:.4f}] {objective(x):>10.4f} {constraint(x):>10.4f}")
    
    print(f"\nAnalytical solution: x = y = 0.5, f = 0.5")
    
constrained_example()

### 4.2 Lagrangian Method

For equality constraints $h(x) = 0$:

$$\mathcal{L}(x, \lambda) = f(x) + \lambda^T h(x)$$

Solve $\nabla_x \mathcal{L} = 0$ and $h(x) = 0$.

In [ ]:
def lagrangian_example():
    """Minimize x² + y² subject to x + y = 1."""
    
    def lagrangian(params):
        x, y, lam = params[0], params[1], params[2]
        f = x**2 + y**2
        h = x + y - 1  # Equality constraint
        return f + lam * h
    
    def kkt_conditions(params):
        """KKT conditions: ∇L = 0."""
        return grad(lagrangian)(params)
    
    # Solve using Newton's method on KKT conditions
    params = jnp.array([0.3, 0.3, 0.0])  # [x, y, λ]
    
    for _ in range(10):
        g = kkt_conditions(params)
        H = jacfwd(kkt_conditions)(params)
        dx = jnp.linalg.solve(H, -g)
        params = params + dx
    
    x, y, lam = params
    print("Lagrangian method:")
    print(f"  x = {x:.6f}, y = {y:.6f}")
    print(f"  λ = {lam:.6f}")
    print(f"  f(x,y) = {x**2 + y**2:.6f}")
    print(f"  Constraint: x + y = {x + y:.6f}")
    print(f"\n  Analytical: x = y = 0.5, λ = 1")

lagrangian_example()

## 5. Root Finding

Newton's method for finding roots of $f(x) = 0$:

$$x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}$$

In [ ]:
def newton_root(f, x0, tol=1e-10, max_iter=100):
    """Find root of f(x) = 0 using Newton's method."""
    df = grad(f)
    x = x0
    
    for i in range(max_iter):
        fx = f(x)
        if jnp.abs(fx) < tol:
            print(f"Converged in {i} iterations")
            break
        x = x - fx / df(x)
    
    return x

# Example: Find √2 by solving x² - 2 = 0
f = lambda x: x**2 - 2
root = newton_root(f, 1.0)
print(f"Root: {root}")
print(f"√2 = {jnp.sqrt(2.0)}")

In [ ]:
# Multidimensional root finding
def newton_root_nd(F, x0, tol=1e-10, max_iter=100):
    """Find root of F(x) = 0 for vector-valued F."""
    J = jacfwd(F)
    x = x0
    
    for i in range(max_iter):
        Fx = F(x)
        if jnp.linalg.norm(Fx) < tol:
            print(f"Converged in {i} iterations")
            break
        Jx = J(x)
        dx = jnp.linalg.solve(Jx, -Fx)
        x = x + dx
    
    return x

# Example: Solve nonlinear system
# x² + y² = 1
# x - y = 0
def system(z):
    x, y = z[0], z[1]
    return jnp.array([
        x**2 + y**2 - 1,
        x - y
    ])

root = newton_root_nd(system, jnp.array([0.5, 0.5]))
print(f"Root: {root}")
print(f"Expected: [1/√2, 1/√2] = [{1/jnp.sqrt(2):.6f}, {1/jnp.sqrt(2):.6f}]")

## 6. Implicit Differentiation

When $y = \text{solve}(f, x)$ where $g(x, y) = 0$, we can compute $dy/dx$ implicitly:

$$\frac{dy}{dx} = -\left(\frac{\partial g}{\partial y}\right)^{-1} \frac{\partial g}{\partial x}$$

In [ ]:
from jax import custom_vjp

@custom_vjp
def implicit_solve(a):
    """Solve x³ + ax - 1 = 0 for x.
    
    Returns the root x as a function of parameter a.
    """
    # Initial guess
    x = jnp.ones(())
    
    # Newton iterations
    for _ in range(20):
        f = x**3 + a*x - 1
        df = 3*x**2 + a
        x = x - f / df
    
    return x

def implicit_solve_fwd(a):
    x = implicit_solve(a)
    return x, (a, x)

def implicit_solve_bwd(res, g):
    a, x = res
    # g(a, x) = x³ + ax - 1 = 0
    # dg/da = x
    # dg/dx = 3x² + a
    # dx/da = -(dg/dx)^{-1} * dg/da = -x / (3x² + a)
    dx_da = -x / (3*x**2 + a)
    return (g * dx_da,)

implicit_solve.defvjp(implicit_solve_fwd, implicit_solve_bwd)

# Test
a = 2.0
x = implicit_solve(a)
dx_da = grad(implicit_solve)(a)

print(f"For a = {a}:")
print(f"  Solution x = {x:.6f}")
print(f"  Verify: x³ + ax - 1 = {x**3 + a*x - 1:.2e}")
print(f"  dx/da = {dx_da:.6f}")
print(f"  Expected: -x/(3x²+a) = {-x/(3*x**2+a):.6f}")

## 7. Using Optax (Production Optimizers)

Optax is JAX's official optimization library with many algorithms.

In [ ]:
try:
    import optax
    
    def train_with_optax():
        # Objective: Rosenbrock
        params = jnp.array([-1.0, 1.0])
        
        # Create optimizer
        optimizer = optax.adam(learning_rate=0.01)
        opt_state = optimizer.init(params)
        
        @jit
        def step(params, opt_state):
            loss, grads = jax.value_and_grad(rosenbrock)(params)
            updates, opt_state = optimizer.update(grads, opt_state)
            params = optax.apply_updates(params, updates)
            return params, opt_state, loss
        
        # Training loop
        for i in range(2000):
            params, opt_state, loss = step(params, opt_state)
            if (i + 1) % 500 == 0:
                print(f"Step {i+1}: loss = {loss:.6f}, params = {params}")
        
        print(f"\nFinal: {params}")
    
    train_with_optax()
    
except ImportError:
    print("Optax not installed. Install with: pip install optax")

## Summary

| Method | Use Case | Pros | Cons |
|--------|----------|------|------|
| Gradient Descent | General | Simple, robust | Slow convergence |
| Momentum | Deep learning | Accelerated | Hyperparameter tuning |
| Adam | Deep learning | Adaptive LR | Memory overhead |
| Newton | Small problems | Quadratic convergence | Hessian expensive |
| Penalty Method | Constraints | Simple | Inexact constraints |
| Lagrangian | Equality constraints | Exact solution | More complex |

**Key Insight**: JAX's autodiff makes all these methods easy to implement and combine!